# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

Before running this notebook in a fresh Google Colab session, please upload the provided assignment data files into the current Colab working directory:

- `train-claims.json`
- `dev-claims.json`
- `test-claims-unlabelled.json`
- `dev-claims-baseline.json`
- `evidence.md`

The full `evidence.json` file is too large for reliable browser upload, so the notebook downloads it automatically into the current working directory from the Google Drive link contained in `evidence.md` when it is not already present.

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*


# 1. DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Gold Evidence Preparation for DeBERTa SFT
This section joins each labelled claim with its gold evidence passages from `evidence.json`, then writes compact JSONL files for supervised fine-tuning a `microsoft/deberta-v3-small` claim classifier.


In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path
from typing import Any

# Constants and paths
WORK_DIR = Path(".")
EVIDENCE_PATH = WORK_DIR / "evidence.json"
TRAIN_CLAIMS_PATH = WORK_DIR / "train-claims.json"
DEV_CLAIMS_PATH = WORK_DIR / "dev-claims.json"
TEST_CLAIMS_PATH = WORK_DIR / "test-claims-unlabelled.json"
DEV_BASELINE_PATH = WORK_DIR / "dev-claims-baseline.json"
EVIDENCE_README_PATH = WORK_DIR / "evidence.md"
PROCESSED_DIR = WORK_DIR / "processed" / "deberta_gold_sft"

LABELS = ("SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED")
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)} # Map labels to integer IDs
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()} # Map integer IDs back to labels

print("Label mapping:", LABEL_TO_ID)


Label mapping: {'SUPPORTS': 0, 'REFUTES': 1, 'NOT_ENOUGH_INFO': 2, 'DISPUTED': 3}


### 1.1.1 Data File Check and Evidence Download in Colab
Upload the small assignment files directly into the current Colab directory. Because `evidence.json` is large, this cell reads the official Google Drive URL from `evidence.md` and downloads `evidence.json` automatically when needed.


In [2]:
import importlib.util
import re
import subprocess
import sys

REQUIRED_UPLOADED_DATA_FILES = [
    TRAIN_CLAIMS_PATH,
    DEV_CLAIMS_PATH,
    TEST_CLAIMS_PATH,
    DEV_BASELINE_PATH,
    EVIDENCE_README_PATH,
]

missing_data_files = [path for path in REQUIRED_UPLOADED_DATA_FILES if not path.exists()]
if missing_data_files:
    missing_list = "\n".join(f"- {path.name}" for path in missing_data_files)
    raise FileNotFoundError(
        "Please upload the provided assignment data files directly into the "
        f"current Colab working directory before running this notebook:\n{missing_list}"
    )

# Print the sizes of the uploaded files and evidence.json (if it exists)
for path in REQUIRED_UPLOADED_DATA_FILES:
    print(f"Found {path} ({path.stat().st_size / 1024:.1f} KB)")

# Check for evidence.json, if not found, attempt to download from the official link in evidence.md
if EVIDENCE_PATH.exists():
    size_mb = EVIDENCE_PATH.stat().st_size / 1024**2
    print(f"Found {EVIDENCE_PATH} ({size_mb:.1f} MB)")
else:
    evidence_md = EVIDENCE_README_PATH.read_text(encoding="utf-8") # Read the contents of evidence.md to find the Google Drive link
    urls = re.findall(r"https?://[^\s)]+", evidence_md)
    drive_urls = [url for url in urls if "drive.google.com" in url]
    if not drive_urls:
        raise RuntimeError(
            "Could not find a Google Drive evidence.json link in evidence.md. "
            "Please open evidence.md, download evidence.json manually, and upload it as evidence.json."
        )
    
    # If gdown is not installed, install it to download the file from Google Drive
    if importlib.util.find_spec("gdown") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])

    import gdown

    evidence_url = drive_urls[0]
    print(f"Downloading official evidence.json from {evidence_url}")
    output = gdown.download(evidence_url, str(EVIDENCE_PATH), quiet=False, fuzzy=True)
    if output is None or not EVIDENCE_PATH.exists():
        raise RuntimeError(
            "Automatic evidence.json download failed. Open evidence.md, download "
            "evidence.json from the official link, then upload it as evidence.json."
        )
    size_mb = EVIDENCE_PATH.stat().st_size / 1024**2
    print(f"Downloaded {EVIDENCE_PATH} ({size_mb:.1f} MB)") 


Found train-claims.json (320.4 KB)
Found dev-claims.json (40.5 KB)
Found test-claims-unlabelled.json (23.8 KB)
Found dev-claims-baseline.json (48.5 KB)
Found evidence.md (0.3 KB)


Downloading...
From (original): https://drive.google.com/uc?id=1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6
From (redirected): https://drive.google.com/uc?id=1JlUzRufknsHzKzvrEjgw8D3n_IRpjzo6&confirm=t&uuid=8073211b-a517-47d7-a796-2c7404f79cc1
To: /content/evidence.json
100%|██████████| 174M/174M [00:02<00:00, 74.2MB/s] 

Downloaded evidence.json (166.1 MB)


In [3]:
def load_json(path: Path) -> dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise ValueError(f"{path} must contain a JSON object.")
    return data


def write_json(path: Path, data: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, sort_keys=True)
        f.write("\n")


def write_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    """ Writes a list of dictionaries to a JSONL file, one JSON object per line. """
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=True) + "\n")


def format_gold_evidence(evidence_ids: list[str], evidence: dict[str, str]) -> str:
    """ Formats the gold evidence for a claim by concatenating the evidence texts with labels."""
    passages = []
    for idx, evidence_id in enumerate(evidence_ids, start=1):
        if evidence_id not in evidence:
            raise KeyError(f"Missing {evidence_id} in evidence.json")
        passages.append(f"Evidence {idx}: {evidence[evidence_id]}")
    return "\n".join(passages)


def build_gold_sft_rows(
    claims: dict[str, Any], evidence: dict[str, str]
) -> list[dict[str, Any]]:
    """ Builds a list of rows for supervised fine-tuning, where each row contains the claim, gold evidence, and label. """
    rows = []
    for claim_id, claim in sorted(claims.items()):
        label = claim.get("claim_label")
        if label not in LABEL_TO_ID:
            raise ValueError(f"{claim_id} has unsupported label: {label!r}")

        evidence_ids = claim.get("evidences")
        if not isinstance(evidence_ids, list) or not evidence_ids:
            raise ValueError(f"{claim_id} must have a non-empty evidences list.")

        claim_text = claim["claim_text"]
        evidence_text = format_gold_evidence(evidence_ids, evidence)
        label_id = LABEL_TO_ID[label]
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim_text,
                "evidence_ids": evidence_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim_text}\nEvidence:\n{evidence_text}",
                "label": label,
                "label_id": label_id,
                "labels": label_id,
                "num_evidences": len(evidence_ids),
            }
        )
    return rows


def summarise_rows(rows: list[dict[str, Any]]) -> dict[str, Any]:
    """ Summarises the dataset by counting labels, evidence distribution, and word lengths. """
    label_counts = Counter(row["label"] for row in rows)
    evidence_counts = Counter(row["num_evidences"] for row in rows)
    claim_word_lengths = [len(row["claim_text"].split()) for row in rows]
    evidence_word_lengths = [len(row["evidence_text"].split()) for row in rows]
    return {
        "num_examples": len(rows),
        "label_counts": dict(sorted(label_counts.items())),
        "evidence_count_distribution": {
            str(k): evidence_counts[k] for k in sorted(evidence_counts)
        },
        "claim_words": {
            "min": min(claim_word_lengths),
            "max": max(claim_word_lengths),
            "mean": sum(claim_word_lengths) / len(claim_word_lengths),
        },
        "evidence_words": {
            "min": min(evidence_word_lengths),
            "max": max(evidence_word_lengths),
            "mean": sum(evidence_word_lengths) / len(evidence_word_lengths),
        },
    }

In [4]:
if not EVIDENCE_PATH.exists():
    raise FileNotFoundError(
        f"{EVIDENCE_PATH} was not found. Upload evidence.md and run the evidence download cell first."
    )

evidence = load_json(EVIDENCE_PATH)
train_claims = load_json(TRAIN_CLAIMS_PATH)
dev_claims = load_json(DEV_CLAIMS_PATH)

train_rows = build_gold_sft_rows(train_claims, evidence)
dev_rows = build_gold_sft_rows(dev_claims, evidence)

write_jsonl(PROCESSED_DIR / "train.jsonl", train_rows)
write_jsonl(PROCESSED_DIR / "dev.jsonl", dev_rows)
write_json(PROCESSED_DIR / "label_mapping.json", LABEL_TO_ID)
write_json(
    PROCESSED_DIR / "metadata.json",
    {
        "model_target": "microsoft/deberta-v3-small",
        "format": "JSONL; tokenize claim_text as text and evidence_text as text_pair",
        "label_mapping": LABEL_TO_ID,
        "train": summarise_rows(train_rows),
        "dev": summarise_rows(dev_rows),
    },
)

print(f"Loaded {len(evidence):,} evidence passages")
print(f"Wrote {len(train_rows)} train examples -> {PROCESSED_DIR / 'train.jsonl'}")
print(f"Wrote {len(dev_rows)} dev examples -> {PROCESSED_DIR / 'dev.jsonl'}")
print("Train summary:", summarise_rows(train_rows))
print("Dev summary:", summarise_rows(dev_rows))

Loaded 1,208,827 evidence passages
Wrote 1228 train examples -> processed/deberta_gold_sft/train.jsonl
Wrote 154 dev examples -> processed/deberta_gold_sft/dev.jsonl
Train summary: {'num_examples': 1228, 'label_counts': {'DISPUTED': 124, 'NOT_ENOUGH_INFO': 386, 'REFUTES': 199, 'SUPPORTS': 519}, 'evidence_count_distribution': {'1': 210, '2': 223, '3': 191, '4': 127, '5': 477}, 'claim_words': {'min': 4, 'max': 67, 'mean': 20.09771986970684}, 'evidence_words': {'min': 9, 'max': 515, 'mean': 99.44218241042346}}
Dev summary: {'num_examples': 154, 'label_counts': {'DISPUTED': 18, 'NOT_ENOUGH_INFO': 41, 'REFUTES': 27, 'SUPPORTS': 68}, 'evidence_count_distribution': {'1': 31, '2': 29, '3': 26, '4': 16, '5': 52}, 'claim_words': {'min': 4, 'max': 65, 'mean': 21.084415584415584}, 'evidence_words': {'min': 12, 'max': 290, 'mean': 92.33766233766234}}


In [5]:
# Inspect one prepared example before training.
train_rows[0]

{'claim_id': 'claim-0',
 'claim_text': 'Global warming is driving polar bears toward extinction',
 'evidence_ids': ['evidence-754568', 'evidence-914173'],
 'evidence_text': 'Evidence 1: Environmental impacts include the extinction or relocation of many species as their ecosystems change, most immediately the environments of coral reefs, mountains, and the Arctic.\nEvidence 2: Rising global temperatures, caused by the greenhouse effect, contribute to habitat destruction, endangering various species, such as the polar bear.',
 'text': 'Claim: Global warming is driving polar bears toward extinction\nEvidence:\nEvidence 1: Environmental impacts include the extinction or relocation of many species as their ecosystems change, most immediately the environments of coral reefs, mountains, and the Arctic.\nEvidence 2: Rising global temperatures, caused by the greenhouse effect, contribute to habitat destruction, endangering various species, such as the polar bear.',
 'label': 'SUPPORTS',
 'label

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## DeBERTa-v3-small Supervised Fine-Tuning Dataset
The following cells load the generated JSONL files and tokenize `(claim_text, evidence_text)` as a sentence-pair input for a 4-way claim-label classifier.

In [6]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "datasets": "datasets",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "evaluate": "evaluate",
    "sentencepiece": "sentencepiece",
    "rank_bm25": "rank-bm25",
    "sentence_transformers": "sentence-transformers",
    "sklearn": "scikit-learn",
    "tqdm": "tqdm",
}
missing_packages = [
    package_name
    for import_name, package_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages]
    )

print("Missing packages installed:" if missing_packages else "All required packages available.", missing_packages)


Missing packages installed: ['evaluate', 'rank-bm25']


## PyTorch and GPU Setup
This cell configures reproducibility and GPU-friendly PyTorch settings for Google Colab T4 training.

In [7]:
import os
import random

import numpy as np
import torch

SEED = 42   # Set a fixed random seed for reproducibility
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disable parallelism in tokenizers to avoid warnings and ensure reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = DEVICE.type == "cuda"

# Google Colab's GPU T4 supports fp16 mixed precision. AMP requires the model weights to stay in
# float32; the Trainer cell below enforces that before training starts.
MIXED_PRECISION = "fp16" if USE_CUDA else "no"  # change to "no" if fp16 is unstable
USE_FP16 = USE_CUDA and MIXED_PRECISION == "fp16"
USE_BF16 = USE_CUDA and MIXED_PRECISION == "bf16" and torch.cuda.is_bf16_supported()

# Safe defaults for Colab T4. Increase train batch size to 16 if memory allows.
PER_DEVICE_TRAIN_BATCH_SIZE = 8 if USE_CUDA else 2
PER_DEVICE_EVAL_BATCH_SIZE = 16 if USE_CUDA else 4
GRADIENT_ACCUMULATION_STEPS = 2 if USE_CUDA else 1 

print("PyTorch version:", torch.__version__)
print("Device:", DEVICE)
if USE_CUDA:
    print("GPU:", torch.cuda.get_device_name(0))
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {total_gb:.1f} GB")
print("mixed precision:", MIXED_PRECISION)
print("fp16 enabled:", USE_FP16)
print("bf16 enabled:", USE_BF16)
print("Effective train batch size:", PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

PyTorch version: 2.10.0+cu128
Device: cuda
GPU: Tesla T4
GPU memory: 14.6 GB
mixed precision: fp16
fp16 enabled: True
bf16 enabled: False
Effective train batch size: 16


In [8]:
from datasets import load_dataset
from transformers import AutoTokenizer

MODEL_NAME = "microsoft/deberta-v3-small"
MAX_LENGTH = 512 # DeBERTa's max input length is 512 tokens. Longer inputs will be truncated.

dataset = load_dataset(
    "json",
    data_files={
        "train": str(PROCESSED_DIR / "train.jsonl"),
        "validation": str(PROCESSED_DIR / "dev.jsonl"),
    },
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def tokenize_for_deberta(batch):
    """ Tokenizes a batch of examples for DeBERTa, using claim_text as the first sequence and evidence_text as the second sequence. """
    return tokenizer(
        batch["claim_text"],
        batch["evidence_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )


tokenized_dataset = dataset.map(
    tokenize_for_deberta,
    batched=True,
    # The original columns are no longer needed after tokenization, and removing them can save memory and speed up training.
    remove_columns=[ 
        "claim_id",
        "claim_text",
        "evidence_ids",
        "evidence_text",
        "text",
        "label",
        "label_id",
        "num_evidences",
    ],
)

tokenized_dataset.set_format("torch")
tokenized_dataset

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Map:   0%|          | 0/1228 [00:00<?, ? examples/s]

Map:   0%|          | 0/154 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1228
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 154
    })
})

In [9]:
# The HuggingFace Trainer expects the target column to be named `labels`.
example = tokenized_dataset["train"][0]
print({key: example[key] for key in ["labels", "input_ids", "attention_mask"]})
print("Decoded example:")
print(tokenizer.decode(example["input_ids"][:160]))

{'labels': tensor(0), 'input_ids': tensor([ 2974,  6965,   269,  1785, 14617,  8660,  1955, 17646, 12993,   376,
          294,  5732,  6598,   680,   262, 17646,   289, 14598,   265,   386,
         2260,   283,   308, 16819,   575,   261,   370,  1587,   262,  5342,
          265, 12920, 24456,   261,  5037,   261,   263,   262, 11069,   260,
        12993,   392,   294, 15237,  1307,  4743,   261,  2044,   293,   262,
        10326,  1290,   261,  3649,   264,  9443,  6138,   261, 48467,   847,
         2260,   261,   405,   283,   262, 14617,  3872,   260]), 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])}
Decoded example:
Global warming is driving polar bears toward extinction Evidence 1: Environmental impacts include the extinction or relocation of many species as their ecosystems cha

In [10]:
import inspect
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

# Load the pre-trained DeBERTa model for sequence classification, specifying the number of labels and the label mappings.
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID_TO_LABEL,
    label2id=LABEL_TO_ID,
    torch_dtype=torch.float32,
)

# Important for AMP/fp16 training: keep trainable parameters in float32.
# Otherwise GradScaler can fail with "Attempting to unscale FP16 gradients".
model.float()
model.to(DEVICE)
trainable_dtypes = sorted({str(p.dtype) for p in model.parameters() if p.requires_grad})
print("Trainable parameter dtypes:", trainable_dtypes)

# Dynamic padding avoids wasting GPU work on short examples. Padding to a multiple
# of 8 lets T4 tensor cores work efficiently during fp16 training. 
# If not using mixed precision, skip padding to a multiple of 8.
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if USE_FP16 or USE_BF16 else None,
)


def compute_metrics(eval_pred):
    """ Computes accuracy and macro F1 score for the evaluation predictions. """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = float((predictions == labels).mean())

    macro_f1_scores = []
    for label_id in range(len(LABELS)):
        true_positive = int(((predictions == label_id) & (labels == label_id)).sum())
        false_positive = int(((predictions == label_id) & (labels != label_id)).sum())
        false_negative = int(((predictions != label_id) & (labels == label_id)).sum())
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if precision + recall else 0.0
        macro_f1_scores.append(f1)

    return {
        "accuracy": accuracy,
        "macro_f1": float(np.mean(macro_f1_scores)),
    }

# Set up the training arguments for the HuggingFace Trainer, including hyperparameters and training configuration.
training_kwargs = {
    "output_dir": "models/deberta_gold_sft",
    "seed": SEED,
    "data_seed": SEED,
    "learning_rate": 2e-5,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": PER_DEVICE_EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "num_train_epochs": 3,
    "weight_decay": 0.01,
    "warmup_steps": 0.06, # warmup_steps
    "logging_steps": 25,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "load_best_model_at_end": True,
    "metric_for_best_model": "macro_f1",
    "greater_is_better": True,
    "fp16": USE_FP16,
    "bf16": USE_BF16,
    "dataloader_pin_memory": USE_CUDA,
    "dataloader_num_workers": 2 if USE_CUDA else 0,
    "group_by_length": True,
    "report_to": "none",
}
# Transformers versions differ here: older versions use `eval_strategy`, newer versions use `evaluation_strategy`.
strategy_arg = (
    "eval_strategy"
    if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters
    else "evaluation_strategy"
)
training_kwargs[strategy_arg] = "epoch"

training_args = TrainingArguments(**training_kwargs)

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_dataset["train"],
    "eval_dataset": tokenized_dataset["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
}

# Transformers versions differ here: older versions use `tokenizer`, newer
# versions use `processing_class`. Detect the available argument so the
# notebook works both locally and on current Colab runtimes.
trainer_init_params = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_init_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_init_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias         

Trainable parameter dtypes: ['torch.float32']


model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

## Train and Evaluate Gold-Evidence DeBERTa Classifier
Run this cell when you are ready to fine-tune. The development split uses gold evidence, so this measures the classifier upper bound when retrieval is perfect.

In [11]:
if USE_CUDA:
    torch.cuda.empty_cache()

try:
    train_result = trainer.train()
except ValueError as exc:
    if "Attempting to unscale FP16 gradients" in str(exc):
        raise RuntimeError(
            "Mixed precision failed because gradients are fp16. Restart the Colab runtime, "
            "rerun the setup cells, and confirm the Trainer cell prints "
            "Trainable parameter dtypes: ['torch.float32']. If it still fails, set "
            "MIXED_PRECISION = 'no' in the PyTorch setup cell and rerun from there."
        ) from exc
    raise

trainer.save_model("models/deberta_gold_sft/best")
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.544207,1.214802,0.441558,0.153153
2,2.236661,1.030075,0.610390,0.358501
3,1.788120,0.937153,0.636364,0.377154


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=231, training_loss=2.2425572737986905, metrics={'train_runtime': 180.8163, 'train_samples_per_second': 20.374, 'train_steps_per_second': 1.278, 'total_flos': 151934378150400.0, 'train_loss': 2.2425572737986905, 'epoch': 3.0})

In [12]:
gold_evidence_dev_metrics = trainer.evaluate(tokenized_dataset["validation"])
print(gold_evidence_dev_metrics)

if USE_CUDA:
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"GPU memory allocated: {allocated:.2f} GB; reserved: {reserved:.2f} GB")

gold_evidence_dev_metrics

{'eval_loss': 0.9371732473373413, 'eval_accuracy': 0.6363636363636364, 'eval_macro_f1': 0.37715412621359223, 'eval_runtime': 1.16, 'eval_samples_per_second': 132.754, 'eval_steps_per_second': 8.62, 'epoch': 3.0}
GPU memory allocated: 1.61 GB; reserved: 4.99 GB


{'eval_loss': 0.9371732473373413,
 'eval_accuracy': 0.6363636363636364,
 'eval_macro_f1': 0.37715412621359223,
 'eval_runtime': 1.16,
 'eval_samples_per_second': 132.754,
 'eval_steps_per_second': 8.62,
 'epoch': 3.0}

## Planned Retrieval Pipeline: BM25 + Dense + RRF + Cross-Encoder
This section implements the retrieval plan used before training the classifier on retrieved evidence. BM25 and dense retrieval independently produce top-100 candidates, Reciprocal Rank Fusion merges them, and a supervised cross-encoder trained with gold evidence plus hard negatives reranks the fused candidates.

In [13]:
import time

from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, InputExample, SentenceTransformer
try:
    from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
except ImportError:
    CEBinaryClassificationEvaluator = None
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

RETRIEVAL_DIR = WORK_DIR / "processed" / "retrieval"
RETRIEVED_SFT_DIR = WORK_DIR / "processed" / "deberta_retrieved_sft_truncated"
RETRIEVAL_DIR.mkdir(parents=True, exist_ok=True)
RETRIEVED_SFT_DIR.mkdir(parents=True, exist_ok=True)

DENSE_RERANKER_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CROSS_ENCODER_OUTPUT_DIR = WORK_DIR / "models" / "cross_encoder_reranker_truncated"

# Truncated retrieval: BM25 supplies a broad lexical pool, MiniLM reranks only
# that pool, then RRF produces the top candidates for the cross-encoder.
BM25_CANDIDATE_K = 500
DENSE_RERANK_K = 500
DENSE_RERANK_BATCH_SIZE = 128
FUSED_CANDIDATE_K = 100
FINAL_EVIDENCE_K = 5
HARD_NEGATIVES_PER_POSITIVE = 5
MAX_HARD_NEGATIVES_PER_CLAIM = 20
TRUNCATED_RETRIEVAL_STRATEGY = "bm25_500_minilm_rerank_rrf_100"

TRAIN_CANDIDATE_CACHE_PATH = RETRIEVAL_DIR / "train_candidate_cache_bm25500_dense_rrf100.json"
DEV_CANDIDATE_CACHE_PATH = RETRIEVAL_DIR / "dev_candidate_cache_bm25500_dense_rrf100.json"

# This run is much cheaper than full-corpus dense retrieval, but still non-trivial.
RUN_FULL_RETRIEVAL_PIPELINE = True

# Set this to True only after changing retrieval hyperparameters or code.
FORCE_REBUILD_CANDIDATE_CACHE = False

print("Truncated retrieval strategy:", TRUNCATED_RETRIEVAL_STRATEGY)
print("Retrieval artifacts will be written under:", RETRIEVAL_DIR)
print("Retrieved-evidence classifier files will be written under:", RETRIEVED_SFT_DIR)


Truncated retrieval strategy: bm25_500_minilm_rerank_rrf_100
Retrieval artifacts will be written under: processed/retrieval
Retrieved-evidence classifier files will be written under: processed/deberta_retrieved_sft_truncated


In [14]:
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]
evidence_id_to_index = {eid: idx for idx, eid in enumerate(evidence_ids)}

_bm25_index = None
_dense_model = None
_cross_encoder_model = None


def bm25_tokenize(text: str) -> list[str]:
    text = text.lower().replace("[", " ").replace("]", " ")
    return re.findall(r"[a-z0-9]+", text)


def ensure_bm25_index() -> BM25Okapi:
    global _bm25_index
    if _bm25_index is None:
        tokenized_evidence = [
            bm25_tokenize(text)
            for text in tqdm(evidence_texts, desc="Tokenising evidence for BM25")
        ]
        _bm25_index = BM25Okapi(tokenized_evidence)
    return _bm25_index


def retrieve_bm25_top_k(claim_text: str, k: int = BM25_CANDIDATE_K) -> list[str]:
    bm25_index = ensure_bm25_index()
    query_tokens = bm25_tokenize(claim_text)
    scores = bm25_index.get_scores(query_tokens)
    k = min(k, len(scores))
    top_indices = np.argpartition(scores, -k)[-k:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return [evidence_ids[i] for i in top_indices]


def get_dense_model() -> SentenceTransformer:
    global _dense_model
    if _dense_model is None:
        _dense_model = SentenceTransformer(DENSE_RERANKER_MODEL_NAME, device=str(DEVICE))
    return _dense_model


def dense_rerank_candidates(
    claim_text: str,
    candidate_ids: list[str],
    batch_size: int = DENSE_RERANK_BATCH_SIZE,
) -> list[str]:
    """Rerank BM25 candidates with MiniLM cosine similarity; no full-corpus dense scan."""
    if not candidate_ids:
        return []

    candidate_ids = candidate_ids[:DENSE_RERANK_K]
    candidate_texts = [evidence[eid] for eid in candidate_ids]
    dense_model = get_dense_model()

    claim_embedding = dense_model.encode(
        claim_text,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)
    candidate_embeddings = dense_model.encode(
        candidate_texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    scores = candidate_embeddings @ claim_embedding
    ranked_indices = np.argsort(scores)[::-1]
    return [candidate_ids[i] for i in ranked_indices]


def reciprocal_rank_fusion(rank_lists: list[list[str]], top_k: int = 100, rrf_k: int = 60) -> list[str]:
    scores = Counter()
    for rank_list in rank_lists:
        for rank, evidence_id in enumerate(rank_list):
            scores[evidence_id] += 1.0 / (rrf_k + rank + 1)
    return [evidence_id for evidence_id, _ in scores.most_common(top_k)]


def retrieve_fused_candidates(claim_text: str, top_k: int = FUSED_CANDIDATE_K) -> list[str]:
    bm25_rank = retrieve_bm25_top_k(claim_text, k=BM25_CANDIDATE_K)
    dense_rank = dense_rerank_candidates(claim_text, bm25_rank)
    return reciprocal_rank_fusion([bm25_rank, dense_rank], top_k=top_k)


def build_candidate_cache(
    claims: dict[str, Any],
    output_path: Path,
) -> dict[str, Any]:
    """Compute BM25, dense-reranked, and fused candidates once for each claim."""
    start_time = time.perf_counter()
    cache: dict[str, Any] = {}

    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        claim_text = claim["claim_text"]
        bm25_rank = retrieve_bm25_top_k(claim_text, k=BM25_CANDIDATE_K)
        dense_rank = dense_rerank_candidates(claim_text, bm25_rank)
        fused_rank = reciprocal_rank_fusion([bm25_rank, dense_rank], top_k=FUSED_CANDIDATE_K)

        cache[claim_id] = {
            "claim_text": claim_text,
            "claim_label": claim.get("claim_label"),
            "gold_evidences": claim.get("evidences", []),
            "bm25_candidates": bm25_rank,
            "dense_reranked_candidates": dense_rank,
            "fused_candidates": fused_rank,
        }

    payload = {
        "metadata": {
            "strategy": TRUNCATED_RETRIEVAL_STRATEGY,
            "bm25_candidate_k": BM25_CANDIDATE_K,
            "dense_rerank_k": DENSE_RERANK_K,
            "fused_candidate_k": FUSED_CANDIDATE_K,
            "dense_reranker_model": DENSE_RERANKER_MODEL_NAME,
            "num_claims": len(cache),
        },
        "claims": cache,
    }
    write_json(output_path, payload)
    elapsed = time.perf_counter() - start_time
    print(f"Wrote candidate cache for {len(cache)} claims -> {output_path}")
    print(f"Candidate cache build time: {elapsed / 60:.2f} min")
    return cache


def load_or_build_candidate_cache(
    claims: dict[str, Any],
    output_path: Path,
    force_rebuild: bool = FORCE_REBUILD_CANDIDATE_CACHE,
) -> dict[str, Any]:
    if output_path.exists() and not force_rebuild:
        payload = load_json(output_path)
        metadata = payload.get("metadata", {})
        if metadata.get("strategy") == TRUNCATED_RETRIEVAL_STRATEGY:
            cache = payload.get("claims", payload)
            print(f"Loaded candidate cache for {len(cache)} claims <- {output_path}")
            return cache
        print(f"Ignoring stale candidate cache with metadata: {metadata}")
    return build_candidate_cache(claims, output_path)


def get_fused_candidates_for_claim(
    claim_id: str,
    claim: dict[str, Any],
    candidate_cache: dict[str, Any] | None = None,
    top_k: int = FUSED_CANDIDATE_K,
) -> list[str]:
    if candidate_cache is not None and claim_id in candidate_cache:
        return candidate_cache[claim_id]["fused_candidates"][:top_k]
    return retrieve_fused_candidates(claim["claim_text"], top_k=top_k)


In [15]:
def build_cross_encoder_rows(
    claims: dict[str, Any],
    output_path: Path,
    candidate_cache: dict[str, Any] | None = None,
    negatives_per_positive: int = HARD_NEGATIVES_PER_POSITIVE,
    max_negatives_per_claim: int = MAX_HARD_NEGATIVES_PER_CLAIM,
) -> list[dict[str, Any]]:
    """ 
        Create claim/evidence relevance rows using gold positives and hard negatives.

        Args:
            claims: A dictionary of claim_id to claim data, where each claim data contains the claim text and a list of gold evidence IDs.
            output_path: The path to write the generated rows as a JSONL file.
            candidate_cache: An optional precomputed cache of candidates for each claim to speed up retrieval. If not provided, candidates will be computed on the fly.
            negatives_per_positive: The number of hard negative examples to include per positive example for each claim.
            max_negatives_per_claim: The maximum number of hard negative examples to include for each claim, regardless of the number of positives. This prevents claims with many gold evidences from dominating the training data with too many negatives.

        Returns:
            A list of dictionaries, where each dictionary represents a claim-evidence pair with the following keys:
                - claim_id: The unique identifier of the claim.
                - claim_text: The text of the claim.
                - evidence_id: The unique identifier of the evidence passage.
                - evidence_text: The text of the evidence passage.
                - label: A binary label indicating relevance (1.0 for gold evidence, 0.0 for hard negatives).
                - source: A string indicating whether the row is a "gold_positive" or a "hard_negative", which can be useful for analysis and debugging.
    """
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        claim_text = claim["claim_text"]
        gold_ids = list(dict.fromkeys(claim["evidences"]))
        gold_set = set(gold_ids)

        for evidence_id in gold_ids:
            rows.append(
                {
                    "claim_id": claim_id,
                    "claim_text": claim_text,
                    "evidence_id": evidence_id,
                    "evidence_text": evidence[evidence_id],
                    "label": 1.0,
                    "source": "gold_positive",
                }
            )

        fused_candidates = get_fused_candidates_for_claim(
            claim_id, claim, candidate_cache=candidate_cache, top_k=FUSED_CANDIDATE_K
        )
        hard_negative_ids = [eid for eid in fused_candidates if eid not in gold_set]
        target_negatives = min(
            len(hard_negative_ids),
            max(max_negatives_per_claim, negatives_per_positive * len(gold_ids)),
        )
        for evidence_id in hard_negative_ids[:target_negatives]:
            rows.append(
                {
                    "claim_id": claim_id,
                    "claim_text": claim_text,
                    "evidence_id": evidence_id,
                    "evidence_text": evidence[evidence_id],
                    "label": 0.0,
                    "source": "hard_negative",
                }
            )

    write_jsonl(output_path, rows)
    print(f"Wrote {len(rows)} cross-encoder rows -> {output_path}")
    return rows


def load_cross_encoder_rows(path: Path) -> list[dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


def rows_to_input_examples(rows: list[dict[str, Any]]) -> list[InputExample]:
    return [
        InputExample(texts=[row["claim_text"], row["evidence_text"]], label=float(row["label"]))
        for row in rows
    ]


def train_cross_encoder_reranker(
    train_rows_path: Path,
    dev_rows_path: Path | None = None,
    epochs: int = 1,
    batch_size: int = 16,
) -> CrossEncoder:
    """
        Trains a cross-encoder on the given training rows, optionally evaluating on dev rows during training.
        
        Args:
        - train_rows_path: Path to the JSONL file containing training rows, where each row is a JSON object with keys "claim_text", "evidence_text", and "label".
        - dev_rows_path: Optional path to the JSONL file containing dev rows for evaluation during training. If provided, the model will be evaluated on these rows at the end of each epoch, and the best model will be saved based on dev performance.
        - epochs: The number of training epochs.
        - batch_size: The training batch size. Adjust based on your GPU memory; smaller batch sizes may be needed for mixed precision training or if your GPU has limited memory.
        Returns:
        - The trained CrossEncoder model.
    """
    train_rows = load_cross_encoder_rows(train_rows_path)
    train_samples = rows_to_input_examples(train_rows)
    train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=batch_size)

    evaluator = None
    if CEBinaryClassificationEvaluator is not None and dev_rows_path is not None and dev_rows_path.exists():
        dev_rows = load_cross_encoder_rows(dev_rows_path)
        dev_samples = rows_to_input_examples(dev_rows)
        evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_samples, name="dev")

    cross_encoder = CrossEncoder(
        CROSS_ENCODER_MODEL_NAME,
        num_labels=1,
        max_length=256,
        device=str(DEVICE),
    )
    warmup_steps = max(1, int(len(train_dataloader) * epochs * 0.1))
    fit_kwargs = {
        "train_dataloader": train_dataloader,
        "evaluator": evaluator,
        "epochs": epochs,
        "warmup_steps": warmup_steps,
        "output_path": str(CROSS_ENCODER_OUTPUT_DIR),
        "show_progress_bar": True,
    }
    if "use_amp" in inspect.signature(cross_encoder.fit).parameters:
        fit_kwargs["use_amp"] = USE_FP16
    cross_encoder.fit(**fit_kwargs)

    # Some sentence-transformers versions only write evaluator outputs during fit,
    # so explicitly save the trained CrossEncoder in a reloadable format.
    CROSS_ENCODER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cross_encoder.save(str(CROSS_ENCODER_OUTPUT_DIR))
    print(f"Saved trained cross-encoder model to {CROSS_ENCODER_OUTPUT_DIR}")

    # Keep using the trained in-memory model for the current Colab session.
    global _cross_encoder_model
    _cross_encoder_model = cross_encoder
    return cross_encoder


def get_cross_encoder_reranker() -> CrossEncoder:
    global _cross_encoder_model
    if _cross_encoder_model is None:
        if CROSS_ENCODER_OUTPUT_DIR.exists():
            try:
                _cross_encoder_model = CrossEncoder(str(CROSS_ENCODER_OUTPUT_DIR), max_length=256, device=str(DEVICE))
            except Exception as exc:
                print(
                    "Could not reload trained cross-encoder from disk; "
                    "falling back to the base cross-encoder. "
                    f"Reason: {exc}"
                )
                _cross_encoder_model = CrossEncoder(CROSS_ENCODER_MODEL_NAME, max_length=256, device=str(DEVICE))
        else:
            _cross_encoder_model = CrossEncoder(CROSS_ENCODER_MODEL_NAME, max_length=256, device=str(DEVICE))
    return _cross_encoder_model


def rerank_with_cross_encoder(
    claim_text: str,
    candidate_ids: list[str],
    cross_encoder: CrossEncoder | None = None,
    batch_size: int = 32,
) -> list[str]:
    if not candidate_ids:
        return []
    cross_encoder = cross_encoder or get_cross_encoder_reranker()
    pairs = [[claim_text, evidence[eid]] for eid in candidate_ids]
    scores = cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    ranked_indices = np.argsort(np.asarray(scores).reshape(-1))[::-1]
    return [candidate_ids[i] for i in ranked_indices]


def retrieve_cross_encoder_top_k(
    claim_text: str,
    final_k: int = FINAL_EVIDENCE_K,
    candidate_k: int = FUSED_CANDIDATE_K,
    cross_encoder: CrossEncoder | None = None,
    candidate_ids: list[str] | None = None,
) -> list[str]:
    fused_candidates = candidate_ids or retrieve_fused_candidates(claim_text, top_k=candidate_k)
    reranked = rerank_with_cross_encoder(claim_text, fused_candidates, cross_encoder=cross_encoder)
    return reranked[:final_k]


In [16]:
def build_retrieved_sft_rows(
    claims: dict[str, Any],
    output_path: Path,
    candidate_cache: dict[str, Any] | None = None,
    final_k: int = FINAL_EVIDENCE_K,
    cross_encoder: CrossEncoder | None = None,
) -> list[dict[str, Any]]:
    """ 
    Create classifier SFT rows using retrieved evidence instead of gold evidence.
    This simulates the input distribution for a retrieved-evidence classifier and can be used to train a DeBERTa model that is more robust to retrieval noise.
        Args:
            claims: The claims dictionary loaded from the original claims.json file.
            output_path: The path where the generated rows will be saved as a JSONL file.
            candidate_cache: An optional precomputed cache of candidates for each claim to speed up retrieval.
            final_k: The number of top evidence passages to retrieve for each claim.
            cross_encoder: An optional trained CrossEncoder model for reranking candidates. If not provided, the function will attempt to load a trained model from disk or fall back to the base cross-encoder model.
        
        returns:
            A list of dictionaries, each representing a claim with its retrieved evidence and associated metadata
    """
    rows = []
    for claim_id, claim in tqdm(claims.items(), desc=f"Building {output_path.name}"):
        fused_candidates = get_fused_candidates_for_claim(
            claim_id, claim, candidate_cache=candidate_cache, top_k=FUSED_CANDIDATE_K
        )
        retrieved_ids = retrieve_cross_encoder_top_k(
            claim["claim_text"],
            final_k=final_k,
            candidate_k=FUSED_CANDIDATE_K,
            cross_encoder=cross_encoder,
            candidate_ids=fused_candidates,
        )
        evidence_text = format_gold_evidence(retrieved_ids, evidence)
        label_id = LABEL_TO_ID[claim["claim_label"]]
        rows.append(
            {
                "claim_id": claim_id,
                "claim_text": claim["claim_text"],
                "evidence_ids": retrieved_ids,
                "evidence_text": evidence_text,
                "text": f"Claim: {claim['claim_text']}\nEvidence:\n{evidence_text}",
                "label": claim["claim_label"],
                "label_id": label_id,
                "labels": label_id,
                "num_evidences": len(retrieved_ids),
            }
        )

    write_jsonl(output_path, rows)
    print(f"Wrote {len(rows)} retrieved-evidence classifier rows -> {output_path}")
    return rows


if RUN_FULL_RETRIEVAL_PIPELINE:
    pipeline_start_time = time.perf_counter()
    cross_encoder_train_path = RETRIEVAL_DIR / "cross_encoder_train_truncated.jsonl"
    cross_encoder_dev_path = RETRIEVAL_DIR / "cross_encoder_dev_truncated.jsonl"

    train_candidate_cache = load_or_build_candidate_cache(train_claims, TRAIN_CANDIDATE_CACHE_PATH)
    dev_candidate_cache = load_or_build_candidate_cache(dev_claims, DEV_CANDIDATE_CACHE_PATH)

    if cross_encoder_train_path.exists() and not FORCE_REBUILD_CANDIDATE_CACHE:
        print(f"Reusing existing {cross_encoder_train_path}")
    else:
        build_cross_encoder_rows(
            train_claims,
            cross_encoder_train_path,
            candidate_cache=train_candidate_cache,
        )

    if cross_encoder_dev_path.exists() and not FORCE_REBUILD_CANDIDATE_CACHE:
        print(f"Reusing existing {cross_encoder_dev_path}")
    else:
        build_cross_encoder_rows(
            dev_claims,
            cross_encoder_dev_path,
            candidate_cache=dev_candidate_cache,
        )

    trained_cross_encoder = train_cross_encoder_reranker(
        cross_encoder_train_path,
        dev_rows_path=cross_encoder_dev_path,
        epochs=1,
        batch_size=16 if USE_CUDA else 4,
    )

    build_retrieved_sft_rows(
        train_claims,
        RETRIEVED_SFT_DIR / "train.jsonl",
        candidate_cache=train_candidate_cache,
        final_k=FINAL_EVIDENCE_K,
        cross_encoder=trained_cross_encoder,
    )
    build_retrieved_sft_rows(
        dev_claims,
        RETRIEVED_SFT_DIR / "dev.jsonl",
        candidate_cache=dev_candidate_cache,
        final_k=FINAL_EVIDENCE_K,
        cross_encoder=trained_cross_encoder,
    )
    pipeline_elapsed = time.perf_counter() - pipeline_start_time
    print(f"Truncated retrieval pipeline elapsed time: {pipeline_elapsed / 60:.2f} min")
else:
    print(
        "Truncated retrieval pipeline is defined but not executed. "
        "Set RUN_FULL_RETRIEVAL_PIPELINE = True to build BM25 candidates, dense-rerank them, "
        "train the cross-encoder with hard negatives, and generate retrieved-evidence SFT files."
    )

Building train_candidate_cache_bm25500_dense_rrf100.json:   0%|          | 0/1228 [00:00<?, ?it/s]

Tokenising evidence for BM25:   0%|          | 0/1208827 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

KeyboardInterrupt: 

## Continue Classifier Training on Retrieved Evidence
After the retrieval pipeline has generated `processed/deberta_retrieved_sft/train.jsonl` and `dev.jsonl`, this cell can continue fine-tuning the claim classifier on realistic retrieved evidence. The gold claim labels are kept unchanged.

In [ ]:
RUN_RETRIEVED_CLASSIFIER_TRAINING = True

retrieved_train_path = RETRIEVED_SFT_DIR / "train.jsonl"
retrieved_dev_path = RETRIEVED_SFT_DIR / "dev.jsonl"

if RUN_RETRIEVED_CLASSIFIER_TRAINING:
    if not retrieved_train_path.exists() or not retrieved_dev_path.exists():
        raise FileNotFoundError(
            "Run the full retrieval pipeline first so retrieved-evidence train/dev JSONL files exist."
        )

    retrieved_dataset = load_dataset(
        "json",
        data_files={
            "train": str(retrieved_train_path),
            "validation": str(retrieved_dev_path),
        },
    )
    tokenized_retrieved_dataset = retrieved_dataset.map(
        tokenize_for_deberta,
        batched=True,
        remove_columns=[
            "claim_id",
            "claim_text",
            "evidence_ids",
            "evidence_text",
            "text",
            "label",
            "label_id",
            "num_evidences",
        ],
    )
    tokenized_retrieved_dataset.set_format("torch")

    classifier_start = "models/deberta_gold_sft/best" if Path("models/deberta_gold_sft/best").exists() else MODEL_NAME
    retrieved_model = AutoModelForSequenceClassification.from_pretrained(
        classifier_start,
        num_labels=len(LABELS),
        id2label=ID_TO_LABEL,
        label2id=LABEL_TO_ID,
        torch_dtype=torch.float32,
    )
    retrieved_model.float()
    retrieved_model.to(DEVICE)

    retrieved_training_kwargs = dict(training_kwargs)
    retrieved_training_kwargs.update(
        {
            "output_dir": "models/deberta_retrieved_sft",
            "learning_rate": 1e-5,
            "num_train_epochs": 3,
        }
    )
    retrieved_training_args = TrainingArguments(**retrieved_training_kwargs)

    retrieved_trainer_kwargs = {
        "model": retrieved_model,
        "args": retrieved_training_args,
        "train_dataset": tokenized_retrieved_dataset["train"],
        "eval_dataset": tokenized_retrieved_dataset["validation"],
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
    }
    if "processing_class" in trainer_init_params:
        retrieved_trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_init_params:
        retrieved_trainer_kwargs["tokenizer"] = tokenizer

    retrieved_trainer = Trainer(**retrieved_trainer_kwargs)
    retrieved_train_result = retrieved_trainer.train()
    retrieved_trainer.save_model("models/deberta_retrieved_sft/best")
    retrieved_evidence_dev_metrics = retrieved_trainer.evaluate(tokenized_retrieved_dataset["validation"])
    print(retrieved_evidence_dev_metrics)
else:
    print(
        "Retrieved-evidence classifier continuation is skipped. "
        "Set RUN_RETRIEVED_CLASSIFIER_TRAINING = True after generating retrieved SFT files."
    )

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*